# BenchmarkTakip - Validasyon
Her hucre bagimsiz calisir. PASS = OK, AssertionError = bug.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'lib'))
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
# TEST: compute_wac - 2 alim, agirlikli ortalama
from portfolio_engine import compute_wac

tx = pd.DataFrame([
    {"Tarih": pd.Timestamp("2023-01-01"), "Varlik Adi": "Altin", "Islem Turu": "ALIS", "Fiyat": 100.0, "Miktar": 2.0, "Komisyon": 0.0},
    {"Tarih": pd.Timestamp("2023-02-01"), "Varlik Adi": "Altin", "Islem Turu": "ALIS", "Fiyat": 200.0, "Miktar": 3.0, "Komisyon": 0.0},
])
tx.columns = ["Tarih", "Varlik Adi", "Islem Turu", "Fiyat", "Miktar", "Komisyon"]
tx = tx.rename(columns={"Varlik Adi": "Varlık Adı", "Islem Turu": "İşlem Türü"})

result = compute_wac(tx)
expected_wac = (100*2 + 200*3) / 5
assert abs(result["Altin"]["wac"] - expected_wac) < 1e-9, f"WAC hatali: {result['Altin']['wac']} != {expected_wac}"
assert result["Altin"]["units"] == 5.0
print(f"PASS: compute_wac - WAC={result['Altin']['wac']:.2f} (beklenen {expected_wac})")

In [ ]:
# TEST: WAC kismi satista sabit kalmali
from portfolio_engine import compute_wac

tx = pd.DataFrame({
    "Tarih": [pd.Timestamp("2023-01-01"), pd.Timestamp("2023-06-01")],
    "Varlık Adı": ["X", "X"],
    "İşlem Türü": ["ALIŞ", "SATIŞ"],
    "Fiyat": [100.0, 150.0],
    "Miktar": [10.0, 3.0],
    "Komisyon": [0.0, 0.0],
})
result = compute_wac(tx)
assert abs(result["X"]["wac"] - 100.0) < 1e-9, "Satis sonrasi WAC degismemeli"
assert result["X"]["units"] == 7.0
assert abs(result["X"]["realized_pnl"] - 150.0) < 1e-9
print("PASS: WAC kismi satista sabit kaliyor")

In [ ]:
# TEST: normalize_to_100 - ilk deger her zaman 100
from benchmark_engine import normalize_to_100

prices = pd.Series(
    [50.0, 75.0, 100.0, 150.0],
    index=pd.date_range("2023-01-02", periods=4, freq="B")
)
result = normalize_to_100(prices, start_date="2023-01-02")
assert result.iloc[0] == 100.0, f"Ilk deger 100 olmali, {result.iloc[0]} geldi"
assert abs(result.iloc[3] - 300.0) < 1e-9
print(f"PASS: normalize_to_100 - {result.tolist()}")

In [ ]:
# TEST: normalize_to_100 - start_date indexte yok, nearest-forward kullanilmali
from benchmark_engine import normalize_to_100

prices = pd.Series(
    [200.0, 400.0],
    index=pd.to_datetime(["2023-01-04", "2023-01-05"])
)
result = normalize_to_100(prices, start_date="2023-01-02")
assert result.iloc[0] == 100.0
assert abs(result.iloc[1] - 200.0) < 1e-9
print("PASS: normalize_to_100 nearest-forward")

In [ ]:
# TEST: compute_real_return_series - CPI kadar buyuyen seri -> duz 100
from portfolio_engine import compute_real_return_series

idx = pd.date_range("2023-01-01", periods=12, freq="MS")
cpi = pd.Series([100.0 * (1.03 ** i) for i in range(12)], index=idx)
nominal = pd.Series([100.0 * (1.03 ** i) for i in range(12)], index=idx)

real = compute_real_return_series(nominal, cpi)
assert abs(real.iloc[0] - 100.0) < 1e-6
assert abs(real.iloc[-1] - 100.0) < 1e-6, f"CPI=nominal -> duz 100 olmali, {real.iloc[-1]:.4f} geldi"
print("PASS: compute_real_return_series - CPI=nominal -> duz 100")

In [ ]:
# TEST: FX donusum yonu - GC=F (USD/oz) x USDTRY = TL/oz
from benchmark_engine import build_benchmark_series

idx = pd.date_range("2023-01-02", periods=5, freq="B")
prices = pd.DataFrame({"GC=F": [1800.0] * 5}, index=idx)
fx_usdtry = pd.Series([26.0] * 5, index=idx)

result = build_benchmark_series(
    symbols=["GC=F"],
    start_date="2023-01-02",
    end_date="2023-01-06",
    prices=prices,
    fx_usdtry=fx_usdtry,
    currency="TL",
)
assert all(abs(v - 100.0) < 1e-9 for v in result["GC=F"].dropna())
print("PASS: FX yonu dogru (GC=F x USDTRY = TL/oz)")

In [ ]:
# TEST: build_deposit_series - ilk deger 100, sonraki degerler buyuyor
from benchmark_engine import build_deposit_series

idx = pd.date_range("2023-01-01", periods=365, freq="D")
tcmb = pd.Series([40.0] * 365, index=idx)

result = build_deposit_series(tcmb, start_date="2023-01-02", end_date="2023-12-29")
assert abs(result.iloc[0] - 100.0) < 1e-6
assert result.iloc[-1] > 130
print(f"PASS: build_deposit_series - %40 faiz 1 yil -> {result.iloc[-1]:.1f}")

In [ ]:
print()
print("=" * 50)
print("Tum testler gecti")
print("=" * 50)